# Advanced-3. Scalable Subspace Hamiltonian Generation from Pauli Terms and Samples

This notebook focuses on one specific SQD ingredient:

1. Start from a qubit Hamiltonian written as a **sparse Pauli sum**  
   $$
   H = \sum_\ell c_\ell P_\ell.
   $$

2. Collect **computational-basis samples**  
   $$
   S = \{x_1, x_2, \dots, x_k\}.
   $$

3. Construct the sampled subspace Hamiltonian
   $$
   H_{\mathrm{sub}} = P_S\, H\, P_S
   $$
   **without** materializing the full $2^n \times 2^n$ $H$ matrix.

For a small toy model we will still compare against the full matrix once, only as a validation check.
Instead, we should construct the projected matrix **directly from the Pauli
Hamiltonian and the sampled bitstrings**.

Suppose the Hamiltonian is written as
$$
H = \sum_{\ell} c_\ell P_\ell,
$$
where each $P_\ell$ is a Pauli string such as `IZZI`, `XXYY`, and so on.
If the sampled support is
$$
S = \{x_1, x_2, \dots, x_k\},
$$
then the SQD projected Hamiltonian is the $k\times k$ matrix
$$
(H_{\mathrm{sub}})_{ab}
= \langle x_a | H | x_b \rangle
= \sum_{\ell} c_\ell \langle x_a | P_\ell | x_b \rangle.
$$

The key point is that a Pauli string acts very simply on a computational-basis state:

- `I` leaves the bit unchanged,
- `Z` leaves the bit unchanged and contributes a sign,
- `X` flips the bit,
- `Y` flips the bit and contributes a phase.

So for a basis bitstring $x$,
$$
P_\ell |x\rangle = \omega_\ell(x)\, |x \oplus f_\ell\rangle,
$$
where $f_\ell$ is the flip mask determined by the `X`/`Y` locations, and
$\omega_\ell(x)$ is the phase/sign accumulated from the local Pauli actions.

This immediately gives
$$
\langle x_a | P_\ell | x_b \rangle
=
\omega_\ell(x_b)\,\delta_{x_a,\; x_b \oplus f_\ell}.
$$

That means we never need the full $2^n \times 2^n$ matrix:

- `I`/`Z`-only terms do **not** flip bitstrings, so they contribute to
  **diagonal elements** of the sampled subspace matrix.
- Terms containing `X` or `Y` create **transitions** between sampled bitstrings.
  They contribute only when the flipped bitstring is also present in the sampled support.

This is the scalable SQD construction: iterate over sampled basis states and Pauli
terms, and fill only the matrix elements that are actually connected inside the
sampled subspace.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.circuit.library import excitation_preserving
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from qiskit.quantum_info import SparsePauliOp, Statevector


## A3-1. Hamiltonian Definition

We use the same 4-qubit qubit Hamiltonian for $\mathrm{H}_2$ as in the earlier tutorial:
$$
H=\sum_j c_j P_j,
\qquad
P_j \in \{I,X,Y,Z\}^{\otimes 4}.
$$

This is still small enough to validate against the full matrix, but the construction we implement below is **not tied** to the small-system regime.


In [2]:
H2_ham = SparsePauliOp(
    [
        'IIII', 'IIIZ', 'IIZI', 'IZII',
        'ZIII', 'IIZZ', 'YXXY', 'XXYY',
        'YYXX', 'XYYX', 'IZIZ', 'ZIIZ',
        'IZZI', 'ZIZI', 'ZZII'
    ],
    coeffs=[
        -0.09886397+0.j,  0.17119775+0.j,  0.17119775+0.j, -0.22278593+0.j,
        -0.22278593+0.j,  0.16862219+0.j,  0.0453222 +0.j, -0.0453222 +0.j,
        -0.0453222 +0.j,  0.0453222 +0.j,  0.12054482+0.j,  0.16586702+0.j,
         0.16586702+0.j,  0.12054482+0.j,  0.17434844+0.j
    ]
)

n_qubits = H2_ham.num_qubits
print("Number of qubits:", n_qubits)
print("Number of Pauli terms:", len(H2_ham))
print(H2_ham)

full_H_matrix = H2_ham.to_matrix() # This is not scalable.

Number of qubits: 4
Number of Pauli terms: 15
SparsePauliOp(['IIII', 'IIIZ', 'IIZI', 'IZII', 'ZIII', 'IIZZ', 'YXXY', 'XXYY', 'YYXX', 'XYYX', 'IZIZ', 'ZIIZ', 'IZZI', 'ZIZI', 'ZZII'],
              coeffs=[-0.09886397+0.j,  0.17119775+0.j,  0.17119775+0.j, -0.22278593+0.j,
 -0.22278593+0.j,  0.16862219+0.j,  0.0453222 +0.j, -0.0453222 +0.j,
 -0.0453222 +0.j,  0.0453222 +0.j,  0.12054482+0.j,  0.16586702+0.j,
  0.16586702+0.j,  0.12054482+0.j,  0.17434844+0.j])


## A3-2. Action of a Pauli String on a Computational-Basis State

Let $x\in\{0,1\}^n$ be a computational-basis bitstring.  
For each qubit, the local Pauli action is:

- $I\ket{b} = \ket{b}$,
- $Z\ket{b} = (-1)^b\ket{b}$,
- $X\ket{b} = \ket{b\oplus 1}$,
- $Y\ket{b} = i(-1)^b\ket{b\oplus 1}$.

So for a full Pauli string $P_\ell$ we have
$$
P_\ell \ket{x}
=
\omega_\ell(x)\,
\ket{x \oplus f_\ell},
$$
where:

- $f_\ell$ is the **flip mask** determined by the positions of `X` and `Y`,
- $\omega_\ell(x)$ is the accumulated sign/phase.

This immediately implies
$$
\langle x_a | P_\ell | x_b \rangle
=
\omega_\ell(x_b)\,\delta_{x_a,\;x_b\oplus f_\ell}.
$$

That is the key rule behind direct subspace construction.

### Diagonal vs transition terms

This rule also explains the difference between two classes of Pauli terms:

1. **Diagonal terms**  
   If a Pauli string contains only `I` and `Z`, then it does not flip the bitstring.
   It contributes only to
   $$
   \langle x|P_\ell|x\rangle.
   $$

2. **Transition terms**  
   If a Pauli string contains `X` or `Y`, then it flips one or more bits.
   It contributes only between connected sampled states
   $$
   x_b \longrightarrow x_b \oplus f_\ell.
   $$

So the sampled subspace Hamiltonian can be filled by checking whether the Pauli-induced destination bitstring is also contained in the sampled support.


In [3]:
def pauli_action_on_bitstring(pauli_label, bitstr):
    """Return (output_bitstring, phase) for P|bitstr>.

    Convention:
    - `bitstr` is the displayed computational-basis bitstring, e.g. '1010'.
    - `pauli_label` is the displayed Qiskit Pauli label, e.g. 'IZXY'.
    - We apply both left-to-right in the same displayed order.
    """
    out = list(bitstr)
    phase = 1.0 + 0.0j

    for i, (p, b) in enumerate(zip(pauli_label, bitstr)):
        if p == "I":
            continue
        elif p == "Z":
            if b == "1":
                phase *= -1
        elif p == "X":
            out[i] = "0" if b == "1" else "1"
        elif p == "Y":
            phase *= 1j if b == "0" else -1j
            out[i] = "0" if b == "1" else "1"
        else:
            raise ValueError(f"Unexpected Pauli character: {p}")

    return "".join(out), phase


Let us inspect a few individual examples.


In [4]:
examples = [
    ("IIZZ", "0101"),
    ("XXYY", "0101"),
    ("YXXY", "1010"),
    ("ZZII", "1110"),
]

for label, bitstr in examples:
    out, phase = pauli_action_on_bitstring(label, bitstr)
    print(f"{label} |{bitstr}> = ({phase}) |{out}>")


IIZZ |0101> = ((-1+0j)) |0101>
XXYY |0101> = ((1-0j)) |1010>
YXXY |1010> = ((1+0j)) |0101>
ZZII |1110> = ((1-0j)) |1110>


## A3-3. Direct Construction of the Sampled Subspace Hamiltonian

Suppose the sampled support is
$$
S=\{x_1,\dots,x_k\}.
$$

Then the projected matrix elements are
$$
(H_{\mathrm{sub}})_{ab}
=
\langle x_a|H|x_b\rangle
=
\sum_\ell c_\ell \langle x_a|P_\ell|x_b\rangle.
$$

The direct algorithm is therefore:

1. keep an ordered list of sampled bitstrings,
2. create a dictionary from bitstring to row/column index,
3. for each Pauli term $c_\ell P_\ell$,
4. for each sampled basis state $x_b$,
5. compute the destination bitstring $x_b \oplus f_\ell$ and phase $\omega_\ell(x_b)$,
6. add $c_\ell \omega_\ell(x_b)$ to the corresponding matrix element **only if** the destination bitstring is also in the sampled support.

This has cost roughly
$$
O\!\left(|S| \times N_{\mathrm{Pauli}}\right),
$$
instead of full-matrix construction.


In [5]:
def build_subspace_hamiltonian_from_samples(hamiltonian, samples, *, return_contributions=False):
    """Build H_sub directly from SparsePauliOp + sampled support.

    Parameters
    ----------
    hamiltonian : SparsePauliOp
        Qubit Hamiltonian written as a Pauli sum.
    samples : iterable[str]
        Sampled computational-basis bitstrings.
    return_contributions : bool
        If True, also return a simple summary of diagonal/transition contributions.

    Returns
    -------
    H_sub : np.ndarray
        Projected Hamiltonian on the sampled support.
    basis : list[str]
        Ordered sampled basis states.
    contribution_log : list[dict], optional
        A lightweight term-by-term summary, useful for teaching/debugging.
    """
    basis = sorted(set(bitstr.replace(" ", "") for bitstr in samples))
    if len(basis) == 0:
        raise ValueError("No sampled bitstrings were provided.")

    index = {b: i for i, b in enumerate(basis)}
    dim = len(basis)
    H_sub = np.zeros((dim, dim), dtype=complex)

    contribution_log = []

    for pauli_label, coeff in zip(hamiltonian.paulis.to_labels(), hamiltonian.coeffs):
        has_transition = any(ch in "XY" for ch in pauli_label)
        hits = 0

        for ket in basis:
            bra, phase = pauli_action_on_bitstring(pauli_label, ket)
            i = index.get(bra)
            if i is not None:
                j = index[ket]
                H_sub[i, j] += coeff * phase
                hits += 1

        if return_contributions:
            contribution_log.append(
                {
                    "label": pauli_label,
                    "coeff": coeff,
                    "type": "transition" if has_transition else "diagonal",
                    "hits_in_support": hits,
                }
            )

    # Clean up tiny non-Hermitian numerical noise if present.
    H_sub = 0.5 * (H_sub + H_sub.conj().T)

    if return_contributions:
        return H_sub, basis, contribution_log
    return H_sub, basis


def sqd_energy_from_samples(samples, hamiltonian):
    """Lowest eigenvalue of the sampled subspace Hamiltonian."""
    H_sub, basis = build_subspace_hamiltonian_from_samples(hamiltonian, samples)
    evals = np.linalg.eigvalsh(H_sub)
    return float(np.min(np.real(evals))), basis, H_sub


## A3-4. A Small Sampled Support by Hand

Before sampling from a circuit, let us manually choose a small support and inspect the resulting projected matrix.


In [6]:
manual_samples = {
    "0011",
    "0110",
    "1001",
    "1100",
}

H_sub_manual, basis_manual, log_manual = build_subspace_hamiltonian_from_samples(
    H2_ham,
    manual_samples,
    return_contributions=True,
)

print("Sampled basis states:", basis_manual)
print("Projected subspace shape:", H_sub_manual.shape)
with np.printoptions(precision=3, suppress=True):
    print(H_sub_manual.real)


Sampled basis states: ['0011', '0110', '1001', '1100']
Projected subspace shape: (4, 4)
[[-1.117  0.     0.     0.181]
 [ 0.    -0.351 -0.181  0.   ]
 [ 0.    -0.181 -0.351  0.   ]
 [ 0.181  0.     0.     0.459]]
